# AI Lab - Practical Question No. 34
## Consistent Heuristic Validator (Monotonicity & Triangle Inequality Analysis)

| Specification | Detail |
| :--- | :--- |
| **Title** | Consistent Heuristic Validator |
| **Syllabus Area** | Informed Search and Heuristics |
| **Difficulty** | 🟠 Advanced |
| **Practical Type** | Validation |
| **Problem Statement** | Write a Python program to check if a given heuristic $h(n)$ for a graph is Consistent (Monotonic). The program must verify the triangle inequality: $h(n) \le c(n, a, n') + h(n')$ for every node and its successors. |
| **Task Requirements** | 1. Iterate through every edge $(u, v)$ with cost $c$.<br>2. Check if $h(u) \le c + h(v)$.<br>3. Output True if consistent, else pinpoint the violating edge. |
| **Input Requirements** | Graph with edge costs, Heuristic dictionary |
| **Expected Output** | Boolean and details of violations |
| **Constraints / Conditions** | Must check all directed edges ($w > 0$) |
| **Core AI Concept** | Consistency / Monotonicity in Heuristics (Triangle Inequality) |
| **Python Concepts Tested** | Nested loops, graph traversal, typing, dataclasses |
| **Expected Algorithm** | Edge validation iteration over all directed edges $\mathcal{O}(E)$ |
| **Recommended Libraries** | Pure Python standard library (None) |
| **Complexity Expectation** | Time Complexity: $\mathcal{O}(E)$, Space Complexity: $\mathcal{O}(V)$ |
| **Sample Input** | $h(A)=5, h(B)=2, \text{cost}(A, B)=2$ |
| **Sample Output** | `False. Violation at edge A->B: 5 > 2 + 2.` |
| **University Practical Importance** | Advanced |
| **Viva Topics** | • Consistency vs Admissibility<br>• Why consistency prevents A* from reopening nodes |
| **Optional Challenge** | Prove programmatically that all consistent heuristics are admissible |

## 1. Theoretical Foundations

### Definition of Consistency (Monotonicity)
In informed search (Russell & Norvig, AIMA 4th Ed.), a heuristic function $h(n)$ is **consistent** (or **monotonic**) if, for every node $n$ and every successor $n'$ generated by any action $a$ with step cost $c(n, a, n')$:

$$h(n) \le c(n, a, n') + h(n')$$

with the standard goal condition $h(\text{Goal}) = 0$.

This is a form of the **triangle inequality**: the estimated cost from node $n$ to the goal cannot be strictly greater than the step cost to a neighbor $n'$ plus the estimated cost from $n'$ to the goal.

### Consistency vs. Admissibility
- **Admissibility**: $h(n) \le h^*(n)$ for all nodes $n$. The heuristic never overestimates the true shortest path distance to the goal.
- **Consistency**: $h(n) \le c(n, a, n') + h(n')$ for every directed edge $(n \to n')$. Local step-wise monotonicity holds everywhere.

**Core Theoretical Theorems:**
1. **Consistency Implies Admissibility**: If a heuristic $h(n)$ is consistent and $h(\text{Goal}) = 0$, then $h(n)$ is **guaranteed to be admissible**. (Proved by path induction).
2. **Admissibility Does NOT Imply Consistency**: A heuristic can underestimate the true distance globally, yet have a localized sharp drop between neighboring nodes that violates the triangle inequality.

### Why Consistency Prevents A* from Reopening Nodes
In **Graph Search** (where an explored/closed set is maintained):
- When $h(n)$ is consistent, the evaluation function $f(n) = g(n) + h(n)$ is **monotonically non-decreasing** along any path:
  $$f(n') = g(n') + h(n') = g(n) + c(n, n') + h(n') \ge g(n) + h(n) = f(n)$$
- Therefore, the sequence of nodes expanded by A* is non-decreasing in $f$-values.
- When A* selects a node $n$ for expansion, the path found to $n$ is **guaranteed to be optimal** ($g(n) = g^*(n)$).
- **No subsequent path will ever discover a cheaper route to an already closed node**.
- Hence, closed nodes **never need to be reopened**, preserving optimal $\mathcal{O}(V)$ graph-search expansions rather than exponential reopenings!

In [ ]:
import sys
from pathlib import Path

# Ensure repository root is on sys.path for direct imports
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.consistent_heuristic_validator import (
    CANONICAL_EXTENDED_GRAPH,
    CANONICAL_EXTENDED_HEURISTIC_CONSISTENT,
    CANONICAL_EXTENDED_HEURISTIC_INCONSISTENT_ADMISSIBLE,
    CANONICAL_SAMPLE_GRAPH,
    CANONICAL_SAMPLE_HEURISTIC_CONSISTENT,
    CANONICAL_SAMPLE_HEURISTIC_VIOLATING,
    SYLLABUS_SAMPLE_OUTPUT,
    ConsistencyProofReport,
    ConsistencyReport,
    EdgeViolation,
    format_consistency_table,
    get_theoretical_analysis,
    get_viva_reference,
    is_consistent_heuristic,
    prove_consistency_implies_admissibility,
    render_ascii_graph,
    validate_consistency,
)

print("Module successfully imported from src.consistent_heuristic_validator!")

## 2. Reproduction of Syllabus Sample Input & Output

We evaluate the exact sample scenario specified by the university syllabus:
- **Edge**: $A \to B$ with step cost $c = 2$
- **Heuristics**: $h(A) = 5, h(B) = 2$
- **Inequality Check**: $h(A) \le c(A, B) + h(B) \implies 5 \le 2 + 2 = 4$ (Violation!)
- **Expected Output**: `False. Violation at edge A->B: 5 > 2 + 2.`

In [ ]:
# Render topology
print(render_ascii_graph(CANONICAL_SAMPLE_GRAPH, CANONICAL_SAMPLE_HEURISTIC_VIOLATING))

# Execute validator
report_sample = validate_consistency(
    CANONICAL_SAMPLE_GRAPH,
    CANONICAL_SAMPLE_HEURISTIC_VIOLATING,
)

print("Validator Summary Output:")
print(report_sample.summary_message)
print("\nSyllabus Match Verification:", report_sample.summary_message == SYLLABUS_SAMPLE_OUTPUT)

## 3. Consistent Heuristic Verification & Formatted Table

Now let us test the same graph with a consistent heuristic: $h(A) = 4, h(B) = 2$.
Here, $h(A) \le c + h(B) \implies 4 \le 2 + 2 = 4$, which satisfies the triangle inequality.

In [ ]:
report_consistent = validate_consistency(
    CANONICAL_SAMPLE_GRAPH,
    CANONICAL_SAMPLE_HEURISTIC_CONSISTENT,
)
print(format_consistency_table(report_consistent))

## 4. Extended Multi-Node Graph Demonstration

Let us explore a multi-node directed graph with 4 nodes ($A, B, C, \text{Goal}$) and multiple paths:
- $A \to B$ (cost 1), $A \to C$ (cost 4)
- $B \to C$ (cost 2), $B \to \text{Goal}$ (cost 5)
- $C \to \text{Goal}$ (cost 1)

In [ ]:
print(render_ascii_graph(CANONICAL_EXTENDED_GRAPH, CANONICAL_EXTENDED_HEURISTIC_CONSISTENT))

# Test 1: Consistent Heuristic
rep_ext_cons = validate_consistency(
    CANONICAL_EXTENDED_GRAPH,
    CANONICAL_EXTENDED_HEURISTIC_CONSISTENT,
)
print(format_consistency_table(rep_ext_cons))

# Test 2: Admissible BUT Inconsistent Heuristic
rep_ext_incons = validate_consistency(
    CANONICAL_EXTENDED_GRAPH,
    CANONICAL_EXTENDED_HEURISTIC_INCONSISTENT_ADMISSIBLE,
)
print(format_consistency_table(rep_ext_incons))

## 5. Optional Challenge: Programmatic Proof (Consistency => Admissibility)

### Mathematical Path Induction
For any node $n_0$, consider the optimal path to the goal: $P = (n_0, n_1, n_2, \dots, n_k = \text{Goal})$ with total cost $h^*(n_0) = \sum_{i=0}^{k-1} c(n_i, n_{i+1})$.

If $h$ is consistent:
$$h(n_0) \le c(n_0, n_1) + h(n_1)$$
$$h(n_1) \le c(n_1, n_2) + h(n_2)$$
$$\dots$$
$$h(n_{k-1}) \le c(n_{k-1}, \text{Goal}) + h(\text{Goal})$$

Summing these $k$ inequalities telescopes:
$$h(n_0) \le \sum_{i=0}^{k-1} c(n_i, n_{i+1}) + h(\text{Goal}) = h^*(n_0) + 0 = h^*(n_0)$$

Let us run the programmatic induction engine to verify this step-by-step for every node in the graph!

In [ ]:
proof_report = prove_consistency_implies_admissibility(
    CANONICAL_EXTENDED_GRAPH,
    goal_node="Goal",
    heuristic=CANONICAL_EXTENDED_HEURISTIC_CONSISTENT,
)
print(proof_report.explanation)

print("\nStep-by-step path induction verification:")
for node, steps in proof_report.telescoping_steps.items():
    h_val = proof_report.heuristics[node]
    h_star = proof_report.true_shortest_costs[node]
    print(f"\nNode [{node}] (h = {h_val:g}, h* = {h_star:g}):")
    if not steps:
        print("  [Terminal or Goal node]")
        continue
    for s in steps:
        print(f"  Step {s.step_index}: {s.inequality_str} {'[Valid]' if s.is_valid else '[Violated]'}")
    print(f"  Induction Conclusion: h({node}) = {h_val:g} <= h*({node}) = {h_star:g} [CONFIRMED ADMISSIBLE]")

## 6. University Viva Voce Examination Guide

In [ ]:
print(get_viva_reference())